In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ")

Libraries loaded 


In [2]:
df = pd.read_csv('../data/walmart_enriched.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("Shape:", df.shape)
print(" Data loaded")

Shape: (420212, 24)
 Data loaded


In [3]:
# Same store and department as Prophet
store1_dept81 = df[(df['Store'] == 1) & (df['Dept'] == 81)].copy()
store1_dept81 = store1_dept81.sort_values('Date').reset_index(drop=True)

print("Total weeks:", len(store1_dept81))
print(store1_dept81[['Date', 'Weekly_Sales']].head())

Total weeks: 143
        Date  Weekly_Sales
0 2010-02-05      30052.75
1 2010-02-12      30694.25
2 2010-02-19      28784.91
3 2010-02-26      28213.79
4 2010-03-05      29957.47


In [4]:
# XGBoost doesn't understand dates naturally. So we'll give it time as numbers + extra features

data = store1_dept81.copy()

# Time features
data['Week_Number']  = data['Date'].dt.isocalendar().week.astype(int)
data['Month']        = data['Date'].dt.month
data['Year']         = data['Date'].dt.year
data['Quarter']      = data['Date'].dt.quarter

# Lag features — what were sales last week, 2 weeks ago, 4 weeks ago
# This is powerful as here the past sales predict future sales
data['Sales_Lag_1']  = data['Weekly_Sales'].shift(1)   # last week
data['Sales_Lag_2']  = data['Weekly_Sales'].shift(2)   # 2 weeks ago
data['Sales_Lag_4']  = data['Weekly_Sales'].shift(4)   # 4 weeks ago

# Rolling average — average sales over last 4 weeks
data['Rolling_Mean_4'] = data['Weekly_Sales'].shift(1).rolling(window=4).mean()

# Drop rows with NaN (caused by lag features)
data = data.dropna()

print("Shape after feature engineering:", data.shape)
print("\nNew features added:")
print(['Week_Number', 'Month', 'Year', 'Quarter',
       'Sales_Lag_1', 'Sales_Lag_2', 'Sales_Lag_4', 'Rolling_Mean_4'])

Shape after feature engineering: (139, 28)

New features added:
['Week_Number', 'Month', 'Year', 'Quarter', 'Sales_Lag_1', 'Sales_Lag_2', 'Sales_Lag_4', 'Rolling_Mean_4']


In [5]:
" define features and target"
# These are the columns XGBoost will learn from
features = [
    'Week_Number',
    'Month',
    'Year',
    'Quarter',
    'IsHoliday',
    'Temperature',
    'Fuel_Price',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'CPI',
    'Unemployment',
    'Size',
    'Sales_Lag_1',
    'Sales_Lag_2',
    'Sales_Lag_4',
    'Rolling_Mean_4'
]

# Target — what we want to predict
target = 'Weekly_Sales'

X = data[features]
y = data[target]

print("Features shape:", X.shape)
print("Target shape  :", y.shape)

Features shape: (139, 19)
Target shape  : (139,)


In [7]:
" train and test data splitting"
# Same 80/20 split as Prophet
split_index = int(len(data) * 0.8)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

Training rows: 111
Testing rows : 28


In [9]:
" training xgboost model"
# Initialize XGBoost model
model_xgb = xgb.XGBRegressor(
    n_estimators      = 500,    # number of trees
    learning_rate     = 0.05,   # how fast it learns
    max_depth         = 5,      # tree depth
    subsample         = 0.8,    # use 80% of data per tree
    colsample_bytree  = 0.8,    # use 80% of features per tree
    random_state      = 42,
    verbosity         = 0
)

# Train
model_xgb.fit(X_train, y_train)

print("XGBoost model trained successfully")

XGBoost model trained successfully


In [10]:
" accuracy"
# Predict on test set
y_pred = model_xgb.predict(X_test)

# Calculate accuracy
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = (abs((y_test - y_pred) / y_test).mean()) * 100

print("=" * 40)
print("     XGBOOST ACCURACY RESULTS")
print("=" * 40)
print(f"  MAE  : ${mae:,.2f}")
print(f"  RMSE : ${rmse:,.2f}")
print(f"  MAPE : {mape:.2f}%")
print(f"  Accuracy : {100 - mape:.2f}%")
print("=" * 40)

     XGBOOST ACCURACY RESULTS
  MAE  : $1,128.50
  RMSE : $1,558.58
  MAPE : 3.60%
  Accuracy : 96.40%


In [11]:
" actuval vs predicted graph plotting"
# Get dates for test period
test_dates = data.iloc[split_index:]['Date']

fig = go.Figure()

# Actual
fig.add_trace(go.Scatter(
    x=test_dates,
    y=y_test,
    mode='lines',
    name='Actual Sales',
    line=dict(color='#2E86AB', width=2)
))

# XGBoost predicted
fig.add_trace(go.Scatter(
    x=test_dates,
    y=y_pred,
    mode='lines',
    name='XGBoost Predicted',
    line=dict(color='#F4A261', width=2, dash='dash')
))

fig.update_layout(
    title='Store 1 Dept 81 — XGBoost Actual vs Predicted',
    xaxis_title='Date',
    yaxis_title='Weekly Sales ($)',
    hovermode='x unified'
)

fig.show()

In [12]:
" feature importance"
# Which features matter most to XGBoost
importance = pd.DataFrame({
    'Feature'   : features,
    'Importance': model_xgb.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Most Important Features:")
print(importance.head(10))

fig = px.bar(
    importance.head(10),
    x='Importance',
    y='Feature',
    orientation='h',
    title='XGBoost — Top 10 Feature Importance',
    color='Importance',
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

Top 10 Most Important Features:
         Feature  Importance
17   Sales_Lag_4    0.160188
13  Unemployment    0.101520
4      IsHoliday    0.094355
9      MarkDown3    0.073350
1          Month    0.070999
12           CPI    0.068905
16   Sales_Lag_2    0.052940
3        Quarter    0.050199
8      MarkDown2    0.048557
0    Week_Number    0.046206


In [13]:
" prophet vs xgboost comparison"
# Load Prophet accuracy for comparison
prophet_mape = 4.16   # your Prophet result from earlier
xgboost_mape = mape   # current XGBoost result

comparison = pd.DataFrame({
    'Model'   : ['Prophet', 'XGBoost'],
    'Accuracy': [100 - prophet_mape, 100 - xgboost_mape],
    'MAPE'    : [prophet_mape, xgboost_mape]
})

print(comparison)

fig = px.bar(
    comparison,
    x='Model',
    y='Accuracy',
    title='Prophet vs XGBoost — Accuracy Comparison',
    color='Model',
    color_discrete_map={
        'Prophet' : '#2E86AB',
        'XGBoost' : '#F4A261'
    },
    text='Accuracy'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.show()

     Model   Accuracy      MAPE
0  Prophet  95.840000  4.160000
1  XGBoost  96.400976  3.599024


In [21]:
" adding holiday saftey buffer as during spikes the model fails"
# Get test period data with dates
test_data = data.iloc[split_index:].copy()
test_data['Predicted_Sales'] = y_pred

# Add smart safety buffer
# 20% buffer on holiday weeks, 3% on normal weeks
test_data['Safety_Buffer'] = np.where(
    test_data['IsHoliday'] == 1,
    1.20,  # holiday weeks
    1.03   # normal weeks
)

# Apply buffer to predictions
test_data['Adjusted_Prediction'] = test_data['Predicted_Sales'] * test_data['Safety_Buffer']

# Recalculate accuracy with adjusted predictions
mae_adj  = mean_absolute_error(test_data['Weekly_Sales'], test_data['Adjusted_Prediction'])
mape_adj = (abs((test_data['Weekly_Sales'] - test_data['Adjusted_Prediction']) / test_data['Weekly_Sales']).mean()) * 100

print("=" * 45)
print("   ACCURACY AFTER HOLIDAY BUFFER APPLIED")
print("=" * 45)
print(f"  Before adjustment accuracy : {100 - mape:.2f}%")
print(f"  After adjustment accuracy  : {100 - mape_adj:.2f}%")
print(f"  MAE before : ${mae:,.2f}")
print(f"  MAE after  : ${mae_adj:,.2f}")
print("=" * 45)

   ACCURACY AFTER HOLIDAY BUFFER APPLIED
  Before adjustment accuracy : 96.40%
  After adjustment accuracy  : 93.90%
  MAE before : $1,128.50
  MAE after  : $1,895.83


In [22]:
" adjusted actual vs predicted graph"
fig = go.Figure()

# Actual
fig.add_trace(go.Scatter(
    x=test_data['Date'],
    y=test_data['Weekly_Sales'],
    mode='lines',
    name='Actual Sales',
    line=dict(color='#2E86AB', width=2)
))

# Original prediction
fig.add_trace(go.Scatter(
    x=test_data['Date'],
    y=test_data['Predicted_Sales'],
    mode='lines',
    name='XGBoost Predicted',
    line=dict(color='#F4A261', width=2, dash='dash')
))

# Adjusted prediction
fig.add_trace(go.Scatter(
    x=test_data['Date'],
    y=test_data['Adjusted_Prediction'],
    mode='lines',
    name='Adjusted Prediction (with buffer)',
    line=dict(color='#2DC653', width=2, dash='dot')
))

fig.update_layout(
    title='Actual vs Predicted vs Adjusted — Holiday Buffer Applied',
    xaxis_title='Date',
    yaxis_title='Weekly Sales ($)',
    hovermode='x unified'
)

fig.show()